# Churn Prediction: Exploratory Data Analysis

## Problem
A telecom company loses revenue when customers cancel, and retention offers cost money. I want to predict which customers are most likely to churn so the company can target offers at the right people instead of everyone.

## Framing
- **Question:** Which customers are likely to churn?
- **Decision it informs:** Who receives a retention offer, given a limited outreach budget.
- **Target variable:** `Churn` (Yes/No).
- **How I'll measure success:** ROC-AUC to judge how well the model ranks customers by risk, plus precision and recall at a chosen threshold to judge the actual outreach list. I won't rely on accuracy, because churn is an imbalanced class and accuracy can look good while the model is useless.
- **Baselines to beat:** (1) predicting "no churn" for every customer, and (2) a plain logistic regression.

## Goals of this notebook
- Check the size, data types, and missing values.
- Measure the churn base rate.
- Look for data quality problems and potential leakage before any modeling.

In [1]:
import pandas as pd

df = pd.read_csv("../data/telco_churn.csv")
print(df.shape)
print()
print(df["Churn"].value_counts(normalize=True))
print()
print(df.dtypes)
print()
print(df.isna().sum())

(7043, 21)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
Streami

In [2]:
# Which rows can't be converted to a number?
as_num = pd.to_numeric(df["TotalCharges"], errors="coerce")
bad = df[as_num.isna()]

print(len(bad), "rows have a non-numeric TotalCharges")
bad[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

11 rows have a non-numeric TotalCharges


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [2]:
# Which rows can't be converted to a number?
as_num = pd.to_numeric(df["TotalCharges"], errors="coerce")
bad = df[as_num.isna()]

print(len(bad), "rows have a non-numeric TotalCharges")
bad[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

11 rows have a non-numeric TotalCharges


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [3]:
# Which rows can't be converted to a number?
as_num = pd.to_numeric(df["TotalCharges"], errors="coerce")
bad = df[as_num.isna()]

print(len(bad), "rows have a non-numeric TotalCharges")
bad[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

11 rows have a non-numeric TotalCharges


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [4]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(df["TotalCharges"].dtype)
print(df["TotalCharges"].isna().sum())

float64
11


In [5]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df = df.drop(columns=["customerID"])

print(df["TotalCharges"].isna().sum(), "missing TotalCharges")
print(df["Churn"].mean().round(4), "overall churn rate")

0 missing TotalCharges
0.2654 overall churn rate


In [6]:
for col in ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling", "SeniorCitizen"]:
    print(df.groupby(col)["Churn"].agg(["mean", "count"]).round(3), "\n")

                 mean  count
Contract                    
Month-to-month  0.427   3875
One year        0.113   1473
Two year        0.028   1695 

                  mean  count
InternetService              
DSL              0.190   2421
Fiber optic      0.419   3096
No               0.074   1526 

                            mean  count
PaymentMethod                          
Bank transfer (automatic)  0.167   1544
Credit card (automatic)    0.152   1522
Electronic check           0.453   2365
Mailed check               0.191   1612 

                   mean  count
PaperlessBilling              
No                0.163   2872
Yes               0.336   4171 

                mean  count
SeniorCitizen              
0              0.236   5901
1              0.417   1142 



df["tenure_band"] = pd.cut(df["tenure"], bins=[-1, 6, 12, 24, 48, 72])
print(df.groupby("tenure_band", observed=True)["Churn"].agg(["mean", "count"]).round(3))

## Findings

**Data quality**
- 7,043 customers, 21 columns, 26.5% churn rate (imbalanced, so accuracy is a poor metric).
- `TotalCharges` was stored as text. 11 rows held blank strings that `isna()` did not detect. All 11 have `tenure = 0`, so they are new customers who have not been billed yet. I filled them with 0 rather than the mean or median, since 0 matches their tenure.
- Dropped `customerID`: it identifies a customer but carries no predictive information.

**Churn patterns (univariate)**
- **Contract:** month-to-month 42.7%, one-year 11.3%, two-year 2.8%.
- **Tenure:** 52.9% in the first 6 months, falling to 9.5% beyond 48 months.
- **Payment method:** electronic check 45.3% versus 15-19% for the other methods.
- **Internet service:** fiber optic 41.9%, DSL 19.0%, no internet 7.4%.
- **Paperless billing:** 33.6% with versus 16.3% without. **Senior citizens:** 41.7% versus 23.6%.

**Caveats and notes for modeling**
- These are marginal rates. Contract, tenure, and internet service overlap, so a single table cannot separate their effects. A multivariate model is needed.
- `TotalCharges` is roughly `tenure × MonthlyCharges`, so these features are strongly correlated. This matters for logistic regression coefficients.
- No obvious target leakage: no column records information that only exists after a customer leaves. I'll recheck this when building features.